# DeepSeek API 大请求体 & 思考模式工具调用测试

本 Notebook 详细记录每一次与服务器的交互，包括：
- 发送的请求体（含大小）
- 服务器的响应内容
- 思考过程 vs 正式回复的区分
- 工具调用的完整流程

In [1]:
import os
import json
import time
import requests
from datetime import datetime
from typing import List, Dict, Any, Optional
from IPython.display import display, HTML

# ============ 配置 ============
API_KEY = os.getenv("VITE_DEEPSEEK_API_KEY", "")
BASE_URL = "https://api.deepseek.com/v1"

# 尝试从 .env 文件读取
if not API_KEY:
    try:
        env_path = "../../../../.env"
        with open(env_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("VITE_DEEPSEEK_API_KEY="):
                    API_KEY = line.strip().split("=", 1)[1]
                    break
        print(f"[配置] 从 {env_path} 读取 API Key")
    except Exception as e:
        print(f"[配置] 读取 .env 失败: {e}")

print(f"[配置] API Key: {API_KEY[:15]}..." if API_KEY else "[配置] ⚠️ 未找到 API Key")
print(f"[配置] Base URL: {BASE_URL}")

[配置] API Key: sk-065e5668a86c...
[配置] Base URL: https://api.deepseek.com/v1


## 辅助函数：详细打印请求和响应

In [2]:
def print_request_details(title: str, payload: dict, model: str):
    """打印请求详情"""
    print(f"\n{'='*60}")
    print(f"📤 {title}")
    print(f"{'='*60}")
    print(f"模型: {model}")
    
    # 计算请求体大小
    payload_json = json.dumps(payload, ensure_ascii=False)
    size_bytes = len(payload_json.encode('utf-8'))
    size_kb = size_bytes / 1024
    print(f"请求体大小: {size_bytes} bytes ({size_kb:.2f} KB)")
    
    # 打印消息列表
    print(f"\n消息数量: {len(payload.get('messages', []))}")
    for i, msg in enumerate(payload.get('messages', [])):
        role = msg.get('role', 'unknown')
        content = msg.get('content', '')
        reasoning = msg.get('reasoning_content', '')
        tool_calls = msg.get('tool_calls')
        tool_call_id = msg.get('tool_call_id')
        
        print(f"\n  [{i}] 角色: {role}")
        if content:
            content_preview = content[:100] + "..." if len(content) > 100 else content
            print(f"      内容: {content_preview}")
        if reasoning:
            reasoning_preview = reasoning[:100] + "..." if len(reasoning) > 100 else reasoning
            print(f"      思考: {reasoning_preview}")
        if tool_calls:
            print(f"      工具调用: {[tc['function']['name'] for tc in tool_calls]}")
        if tool_call_id:
            print(f"      工具调用ID: {tool_call_id}")
    
    # 打印其他参数
    if 'tools' in payload:
        print(f"\n工具数量: {len(payload['tools'])}")
        for tool in payload['tools']:
            print(f"  - {tool['function']['name']}")
    
    if 'thinking' in payload:
        print(f"\n思考模式: {payload['thinking']}")
    
    print(f"\n完整请求体（前500字符）:")
    print(payload_json[:500] + "..." if len(payload_json) > 500 else payload_json)
    
    return size_kb

def print_response_details(title: str, message: dict, elapsed_time: float = None):
    """打印响应详情"""
    print(f"\n{'='*60}")
    print(f"📥 {title}")
    print(f"{'='*60}")
    
    if elapsed_time:
        print(f"响应时间: {elapsed_time:.2f} 秒")
    
    content = message.get('content', '')
    reasoning = message.get('reasoning_content', '')
    tool_calls = message.get('tool_calls')
    finish_reason = message.get('finish_reason')
    
    print(f"\n完成原因: {finish_reason}")
    
    # 思考过程
    if reasoning:
        print(f"\n💭 思考过程 ({len(reasoning)} 字符):")
        print("-" * 40)
        print(reasoning)
        print("-" * 40)
    else:
        print(f"\n💭 思考过程: 无")
    
    # 正式回复
    if content:
        print(f"\n💬 正式回复 ({len(content)} 字符):")
        print("-" * 40)
        print(content)
        print("-" * 40)
    else:
        print(f"\n💬 正式回复: (空)")
    
    # 工具调用
    if tool_calls:
        print(f"\n🔧 工具调用请求 ({len(tool_calls)} 个):")
        for tc in tool_calls:
            print(f"  - 工具名: {tc['function']['name']}")
            print(f"    调用ID: {tc['id']}")
            print(f"    参数: {tc['function']['arguments']}")
    else:
        print(f"\n🔧 工具调用: 无")

def print_tool_execution(tool_name: str, arguments: dict, result: str):
    """打印工具执行详情"""
    print(f"\n{'='*60}")
    print(f"🔨 工具执行: {tool_name}")
    print(f"{'='*60}")
    print(f"输入参数: {json.dumps(arguments, ensure_ascii=False)}")
    print(f"执行结果 ({len(result)} 字符): {result}")


## 测试 1: 普通模式 + 工具调用（详细记录每一步）

In [3]:
print("\n" + "="*60)
print("测试 1: 普通模式 + 工具调用（非流式）")
print("="*60)

model = "deepseek-chat"
messages = [{"role": "user", "content": "现在几点了？"}]

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取当前时间",
            "parameters": {"type": "object", "properties": {}}
        }
    }
]

# ============ 第 1 轮请求 ============
payload_1 = {
    "model": model,
    "messages": messages,
    "tools": tools,
    "max_tokens": 2048,
    "stream": False
}

size_kb = print_request_details("第 1 轮请求 (发送用户问题 + 工具定义)", payload_1, model)

headers = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}
start = time.time()
resp = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload_1)
elapsed_1 = time.time() - start

if resp.status_code != 200:
    print(f"❌ 请求失败: {resp.text}")
else:
    data = resp.json()
    msg_1 = data["choices"][0]["message"]
    
    print_response_details("第 1 轮响应 (服务器返回思考 + 工具调用请求)", msg_1, elapsed_1)
    
    # 检查是否有工具调用
    if msg_1.get("tool_calls"):
        # ============ 执行工具 ============
        tc = msg_1["tool_calls"][0]
        tool_name = tc["function"]["name"]
        tool_args = json.loads(tc["function"]["arguments"] or "{}")
        
        # 模拟执行（实际应该调用真实函数）
        from datetime import datetime
        tool_result = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        print_tool_execution(tool_name, tool_args, tool_result)
        
        # ============ 第 2 轮请求 ============
        # 添加 assistant 消息（含 tool_calls）
        messages.append({
            "role": "assistant",
            "content": msg_1.get("content", ""),
            "tool_calls": msg_1["tool_calls"]
        })
        
        # 添加 tool 消息（含执行结果）
        messages.append({
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": tool_result
        })
        
        payload_2 = {
            "model": model,
            "messages": messages,
            "max_tokens": 2048,
            "stream": False
        }
        
        print_request_details("第 2 轮请求 (发送工具执行结果，等待最终回复)", payload_2, model)
        
        start = time.time()
        resp_2 = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload_2)
        elapsed_2 = time.time() - start
        
        if resp_2.status_code == 200:
            data_2 = resp_2.json()
            msg_2 = data_2["choices"][0]["message"]
            print_response_details("第 2 轮响应 (服务器返回最终回答)", msg_2, elapsed_2)
        else:
            print(f"❌ 第 2 轮请求失败: {resp_2.text}")
    else:
        print(f"\n⚠️ 服务器未请求工具调用")


测试 1: 普通模式 + 工具调用（非流式）

📤 第 1 轮请求 (发送用户问题 + 工具定义)
模型: deepseek-chat
请求体大小: 292 bytes (0.29 KB)

消息数量: 1

  [0] 角色: user
      内容: 现在几点了？

工具数量: 1
  - get_current_time

完整请求体（前500字符）:
{"model": "deepseek-chat", "messages": [{"role": "user", "content": "现在几点了？"}], "tools": [{"type": "function", "function": {"name": "get_current_time", "description": "获取当前时间", "parameters": {"type": "object", "properties": {}}}}], "max_tokens": 2048, "stream": false}

📥 第 1 轮响应 (服务器返回思考 + 工具调用请求)
响应时间: 1.85 秒

完成原因: None

💭 思考过程: 无

💬 正式回复 (11 字符):
----------------------------------------
我来帮您查看当前时间。
----------------------------------------

🔧 工具调用请求 (1 个):
  - 工具名: get_current_time
    调用ID: call_00_G8kFGelxofMk65DzBw3CQsa8
    参数: {}

🔨 工具执行: get_current_time
输入参数: {}
执行结果 (19 字符): 2026-02-23 01:41:55

📤 第 2 轮请求 (发送工具执行结果，等待最终回复)
模型: deepseek-chat
请求体大小: 456 bytes (0.45 KB)

消息数量: 3

  [0] 角色: user
      内容: 现在几点了？

  [1] 角色: assistant
      内容: 我来帮您查看当前时间。
      工具调用: ['get_current_time']

  [2] 角色: t

## 测试 2: 思考模式 (deepseek-reasoner) + 工具调用

In [4]:
print("\n" + "="*60)
print("测试 2: 思考模式 + 工具调用（非流式）")
print("="*60)

model = "deepseek-reasoner"
messages = [{"role": "user", "content": "北京现在天气怎么样？我需要知道温度。"}]

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    }
]

# ============ 第 1 轮请求 ============
payload_1 = {
    "model": model,
    "messages": messages,
    "tools": tools,
    "max_tokens": 2048,
    "stream": False
}

size_kb = print_request_details("第 1 轮请求 (思考模式)", payload_1, model)

start = time.time()
resp = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload_1)
elapsed_1 = time.time() - start

if resp.status_code != 200:
    print(f"❌ 请求失败: {resp.text}")
else:
    data = resp.json()
    msg_1 = data["choices"][0]["message"]
    
    print_response_details("第 1 轮响应", msg_1, elapsed_1)
    
    if msg_1.get("tool_calls"):
        tc = msg_1["tool_calls"][0]
        tool_name = tc["function"]["name"]
        tool_args = json.loads(tc["function"]["arguments"] or "{}")
        tool_result = "北京今天晴天，温度 15-22°C"
        
        print_tool_execution(tool_name, tool_args, tool_result)
        
        # ============ 关键：保留 reasoning_content ============
        print(f"\n[关键] 添加 assistant 消息时保留 reasoning_content")
        
        reasoning_1 = msg_1.get("reasoning_content", "")
        print(f"思考内容长度: {len(reasoning_1)} 字符")
        
        messages.append({
            "role": "assistant",
            "content": msg_1.get("content", ""),
            "reasoning_content": reasoning_1,  # ⚠️ 必须保留！
            "tool_calls": msg_1["tool_calls"]
        })
        
        messages.append({
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": tool_result
        })
        
        payload_2 = {
            "model": model,
            "messages": messages,
            "max_tokens": 2048,
            "stream": False
        }
        
        print_request_details("第 2 轮请求 (含 reasoning_content)", payload_2, model)
        
        start = time.time()
        resp_2 = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload_2)
        elapsed_2 = time.time() - start
        
        if resp_2.status_code == 200:
            data_2 = resp_2.json()
            msg_2 = data_2["choices"][0]["message"]
            
            # 合并思考过程
            reasoning_2 = msg_2.get("reasoning_content", "")
            full_reasoning = reasoning_1 + "\n\n[继续思考]\n" + reasoning_2 if reasoning_2 else reasoning_1
            
            print(f"\n[合并后的思考过程]\n{full_reasoning}")
            print_response_details("第 2 轮响应", msg_2, elapsed_2)
        else:
            print(f"❌ 第 2 轮请求失败: {resp_2.text}")
    else:
        print(f"\n⚠️ 无工具调用")


测试 2: 思考模式 + 工具调用（非流式）

📤 第 1 轮请求 (思考模式)
模型: deepseek-reasoner
请求体大小: 381 bytes (0.37 KB)

消息数量: 1

  [0] 角色: user
      内容: 北京现在天气怎么样？我需要知道温度。

工具数量: 1
  - get_weather

完整请求体（前500字符）:
{"model": "deepseek-reasoner", "messages": [{"role": "user", "content": "北京现在天气怎么样？我需要知道温度。"}], "tools": [{"type": "function", "function": {"name": "get_weather", "description": "获取指定城市天气", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}], "max_tokens": 2048, "stream": false}

📥 第 1 轮响应
响应时间: 2.71 秒

完成原因: None

💭 思考过程 (98 字符):
----------------------------------------
用户想知道北京现在的天气情况，特别是温度信息。我需要使用get_weather工具来获取北京的温度数据。这个工具可以获取指定城市的天气信息，包括温度。那么，我就调用这个工具来获取北京的天气数据吧。
----------------------------------------

💬 正式回复: (空)

🔧 工具调用请求 (1 个):
  - 工具名: get_weather
    调用ID: call_00_NlhYCmjemdPIiVLcZft6igJX
    参数: {"city": "北京"}

🔨 工具执行: get_weather
输入参数: {"city": "北京"}
执行结果 (17 字符): 北京今天晴天，温度 15-22°C

[关键] 添加 assistant 消息时保留 reasoning_content
思考内容长度: 98 字符

📤

## 测试 3: 思考模式流式输出（观察 reasoning_content 和 content 的交替）

In [5]:
print("\n" + "="*60)
print("测试 3: 思考模式流式输出")
print("="*60)

model = "deepseek-reasoner"
messages = [{"role": "user", "content": "解释什么是机器学习，用简单的比喻。"}]

payload = {
    "model": model,
    "messages": messages,
    "max_tokens": 1024,
    "stream": True
}

print_request_details("流式请求", payload, model)

print(f"\n📥 流式响应 (实时显示):\n{'='*60}")

resp = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload, stream=True)

reasoning_buffer = []
content_buffer = []
chunk_count = 0

for line in resp.iter_lines():
    if line:
        line_text = line.decode('utf-8')
        if line_text.startswith('data: '):
            data = line_text[6:]
            if data == '[DONE]':
                break
            try:
                chunk = json.loads(data)
                delta = chunk.get('choices', [{}])[0].get('delta', {})
                
                if delta.get('reasoning_content'):
                    text = delta['reasoning_content']
                    reasoning_buffer.append(text)
                    print(f"[思考] {text}", end='', flush=True)
                
                if delta.get('content'):
                    text = delta['content']
                    content_buffer.append(text)
                    print(f"[回复] {text}", end='', flush=True)
                
                chunk_count += 1
            except:
                pass

print(f"\n\n{'='*60}")
print(f"流式完成!")
print(f"总块数: {chunk_count}")
print(f"思考总长度: {len(''.join(reasoning_buffer))} 字符")
print(f"回复总长度: {len(''.join(content_buffer))} 字符")

print(f"\n💭 完整思考过程:")
print(''.join(reasoning_buffer))

print(f"\n💬 完整回复:")
print(''.join(content_buffer))


测试 3: 思考模式流式输出

📤 流式请求
模型: deepseek-reasoner
请求体大小: 164 bytes (0.16 KB)

消息数量: 1

  [0] 角色: user
      内容: 解释什么是机器学习，用简单的比喻。

完整请求体（前500字符）:
{"model": "deepseek-reasoner", "messages": [{"role": "user", "content": "解释什么是机器学习，用简单的比喻。"}], "max_tokens": 1024, "stream": true}

📥 流式响应 (实时显示):
[思考] 嗯[思考] ，[思考] 用户[思考] 想要[思考] 一个[思考] 关于[思考] 机器[思考] 学习的[思考] 简单[思考] 比喻[思考] 解释[思考] 。[思考] 这[思考] 应该[思考] 是一个[思考] 对[思考] 技术[思考] 概念[思考] 不太[思考] 熟悉的[思考] 初学者[思考] ，[思考] 或者[思考] 可能[思考] 只是想[思考] 快速[思考] 理解[思考] 核心[思考] 思想[思考] 的人[思考] 。[思考] 需要[思考] 避免[思考] 使用[思考] 术语[思考] ，[思考] 用[思考] 日常[思考] 生活中的[思考] 例子[思考] 来[思考] 类比[思考] 。

[思考] 想到[思考] 可以用[思考] “[思考] 学徒[思考] ”[思考] 或[思考] “[思考] 侦探[思考] ”[思考] 这样的[思考] 角色[思考] 来[思考] 比喻[思考] 机器学习[思考] 系统[思考] ，[思考] 这样[思考] 比较[思考] 形象[思考] 。[思考] 重点[思考] 要[思考] 突出[思考] 机器学习[思考] 是从[思考] 例子[思考] 中[思考] 学习[思考] 规律[思考] ，[思考] 而不是[思考] 直接[思考] 编程[思考] 。[思考] 还可以[思考] 对比[思考] 传统[思考] 编程[思考] 来[思考] 帮助[思考] 理解[思考] 。

[思考] 可以[思考] 准备[思考] 一个[思考] 核心[思考] 比喻[思考] ，[思考] 再[思考] 补充[思考] 两个[思考] 不同[思考] 角[思考] 度的[思考] 比喻[思考] ，[思考] 让[思考] 用户[思考] 有多[思考] 元[思

## 测试 4: 大请求体测试 (>90KB)

In [ ]:
print("\n" + "="*60)
print("测试 4: 大请求体 (>90KB)")
print("="*60)

# 创建大请求体
filler = "这是一段用于测试大请求体的重复文本内容。" * 1000  # 约 47KB

model = "deepseek-chat"
messages = [
    {"role": "system", "content": "你是一个 helpful 助手。"},
    {"role": "user", "content": f"请先阅读以下背景信息，然后回答我的问题：\n\n{filler}\n\n问题：请总结以上内容。"}
]

payload = {
    "model": model,
    "messages": messages,
    "max_tokens": 1024,
    "stream": False
}

size_kb = print_request_details("大请求体测试", payload, model)

if size_kb < 99:
    print(f"\n⚠️ 请求体大小 ({size_kb:.2f} KB) 未达到 90KB，添加更多内容...")
    additional = "补充内容。" * 2000
    messages[1]["content"] += f"\n\n{additional}"
    payload["messages"] = messages
    size_kb = print_request_details("调整后的大请求体", payload, model)

start = time.time()
resp = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload)
elapsed = time.time() - start

print(f"\n[结果] 状态码: {resp.status_code}, 耗时: {elapsed:.2f}秒")

if resp.status_code == 200:
    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    print(f"✅ 成功! 回复长度: {len(content)} 字符")
    print(f"回复预览: {content[:200]}...")
else:
    print(f"❌ 失败: {resp.text[:500]}")


测试 4: 大请求体 (>90KB)

📤 大请求体测试
模型: deepseek-chat
请求体大小: 60276 bytes (58.86 KB)

消息数量: 2

  [0] 角色: system
      内容: 你是一个 helpful 助手。

  [1] 角色: user
      内容: 请先阅读以下背景信息，然后回答我的问题：

这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内...

完整请求体（前500字符）:
{"model": "deepseek-chat", "messages": [{"role": "system", "content": "你是一个 helpful 助手。"}, {"role": "user", "content": "请先阅读以下背景信息，然后回答我的问题：\n\n这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文本内容。这是一段用于测试大请求体的重复文...

⚠️ 请求体大小 (58.86 KB) 未达到 90KB，添加更多内容...

📤 调整后的大请求体
模型: deepseek-chat
请求体大小: 90280 bytes (88.16 KB)

消息数量: 2

  [0] 角色: system
      内容: 你是一个 helpful 助手。

  [1] 角色: user
      内容: 请先阅读以下背景信息，然后回答我的问题：

这是一段用于测试大请求体的重复文

## 总结：关键发现

通过以上测试，我们得出以下结论用于前端实现：

### 1. 请求体结构
- **普通模式**: `messages` + `tools` + `max_tokens`
- **思考模式**: 同上，但 `model` 改为 `deepseek-reasoner`

### 2. 响应内容区分
- **思考过程**: `message.reasoning_content` (仅思考模式有)
- **正式回复**: `message.content`
- **工具调用**: `message.tool_calls`

### 3. 工具调用流程
```
第 1 轮: User 提问 → AI 思考 → AI 请求工具 → 执行工具
第 2 轮: 发送工具结果 → AI 思考 → AI 最终回答
```

### 4. 关键注意事项
- **必须保留 reasoning_content**: 第 2 轮请求时，assistant 消息必须包含 `reasoning_content`
- **请求体大小**: API 支持最大 100KB，超过会报错
- **流式输出**: 思考模式下，reasoning_content 和 content 会交替流式输出